# Part A

## 1. Loading the data into pandas dataframes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ratings_df = pd.read_csv('data/ratings.csv')
movies_df = pd.read_csv('data/movies.csv')


: 

## 2. Report

In [ ]:
unique_users = ratings_df['userId'].nunique()
unique_movies = ratings_df['movieId'].nunique()
num_ratings = ratings_df.shape[0]
sparsity = 1 - (num_ratings / (unique_users * unique_movies))

print("Number of unique users:", unique_users)
print("Number of unique movies:", unique_movies)
print("Number of ratings:", num_ratings)
print(f"Sparsity of the ratings matrix: {sparsity:.3%}")

## 3. Building the user-item matrix

In [ ]:
matrix = ratings_df.pivot_table(index='userId', columns='movieId', values='rating')

## 4. Plotting

### 4.1 Distribution of ratings

In [ ]:
rating_counts = ratings_df["rating"].value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar(rating_counts.index, rating_counts.values, width=0.3)
plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.title("Distribution of ratings")
plt.xticks(rating_counts.index)
plt.show()

### 4.2 Distribution of ratings per user

In [ ]:
ratings_per_user = ratings_df.groupby('userId').size()

plt.figure(figsize=(8, 4))
plt.hist(ratings_per_user, bins=50)
plt.xlabel("Number of ratings given by a user")
plt.ylabel("Number of users")
plt.title("Ratings per user")
plt.show()

### Notes:-

1) In the distributions of ratings: 4.0 and 3.0 are the most common ratings, And that the whole numbers always beat the half-steps, People usually approximates to the whole number than using the halfs.

2) In the distribution of ratings per user: most users rated only a small number of movies, while a few users rated over 1000 movies, and one rated about 2700 movies. Since each user covers only a tiny fraction of the catalog, two users often share very few co-rated items.

# Part B - Similarity metrics from scratch


In [ ]:
def get_co_rated(matrix, a, b, axis="index"):
    if axis == "index":
        va, vb = matrix.loc[a], matrix.loc[b] # hadelo 2 rows
    elif axis == "columns":
        va, vb = matrix[a], matrix[b] # hadeelo 2 cols
    else:
        raise ValueError("axis must be 'index' or 'columns'") #wrong choice nigga
    mask = va.notna() & vb.notna() # keep only entries BOTH rated - NaN is never turned into 0
    return va[mask].to_numpy(dtype=float), vb[mask].to_numpy(dtype=float)

def euclidean_distance(vec1, vec2):
    return float(np.sqrt(np.sum((vec1 - vec2) ** 2)))

def euclidean_similarity(vec1, vec2):
    return 1.0 / (1.0 + euclidean_distance(vec1, vec2))

def cosine_similarity(vec1, vec2):
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denom == 0: # zero vector -> angle undefined, treat as no similarity
        return 0.0
    return float(np.dot(vec1, vec2) / denom)

def pearson_similarity(vec1, vec2):
    # mean-center each vector, then take cosine
    return cosine_similarity(vec1 - vec1.mean(), vec2 - vec2.mean())

In [ ]:
# Sanity check against Section 3.2 (own test code)
A = np.array([5, 4, 5, 3.]); B = np.array([3, 2, 3, 1.])
print("euclidean distance  :", euclidean_distance(A, B))     # 4.0
print("euclidean similarity:", euclidean_similarity(A, B))   # 0.2
print("cosine              :", round(cosine_similarity(A, B), 3))   # 0.987
print("pearson             :", pearson_similarity(A, B))     # 1.0

toy = pd.DataFrame({10: [5, 4, np.nan], 20: [3, np.nan, 2], 30: [4, 5, 1], 40: [np.nan, 2, 3]},
                   index=["u1", "u2", "u3"])
print(toy)
print(get_co_rated(toy, "u1", "u2", "index")) # only item 10 and 30 -> ([5,4],[4,5])
print(get_co_rated(toy, 10, 30, "columns")) # only users u1,u2,... where both rated

# Part C

## 9. Find k nearest users

In [ ]:
def find_k_nearest_users(matrix,target_user,k,similarity_fn):
    scores = []
    for user in matrix.index:
        if user != target_user:
            vec1, vec2 = get_co_rated(matrix, target_user, user, axis="index")
            if (len(vec1) < 2):
                continue
            score = similarity_fn(vec1, vec2)
            if np.isnan(score):
                continue
            scores.append((user, score))

    return sorted(scores, key=lambda t: t[1], reverse=True)[:k]



## 10. Predict rating user based

In [ ]:
def predict_rating_user_based(matrix,target_user,target_item,k,similarity_fn):
    if target_item not in matrix.columns:
        return None
    if target_user not in matrix.index:
        return None
    
    neighbors = find_k_nearest_users(matrix, target_user, k, similarity_fn)
    numerator = 0
    denominator = 0
    for user, score in neighbors:
        rating = matrix.loc[user, target_item]
        if np.isnan(rating):
            continue
        numerator += score * rating
        denominator += abs(score)
    if denominator == 0:
        return None
    return numerator / denominator


# Part D

## 11.1 Find k nearest items

In [ ]:
def find_k_nearest_items(matrix, target_item, k, similarity_fn):
    scores = []
    for item in matrix.columns:
        if item != target_item:
            vec1, vec2 = get_co_rated(matrix, target_item, item, axis="columns")
            if (len(vec1) < 2):
                continue
            score = similarity_fn(vec1, vec2)
            if np.isnan(score):
                continue
            scores.append((item, score))
            
    return sorted(scores, key=lambda t: t[1], reverse=True)[:k]



## 11.2 Predict rating item based

In [ ]:
def predict_rating_item_based(matrix,target_user,target_item,k,similarity_fn):
    if target_item not in matrix.columns:
        return None
    if target_user not in matrix.index:
        return None
    
    neighbors = find_k_nearest_items(matrix, target_item, k, similarity_fn)
    numerator = 0
    denominator = 0
    for item, score in neighbors:
        rating = matrix.loc[target_user, item]
        if np.isnan(rating):
            continue
        numerator += score * rating
        denominator += abs(score)
    if denominator == 0:
        return None
    return numerator / denominator


## 12. Explanation:-

We reused the same logic from part C, checking the same structure of the neighbor search (loop, skip himself, skip if fewer than 2 co-rated, applying similarity_fn) and using the same average formula.

We changed only that the loop runs over columns not rows (axis = "columns" instead axis = "index") and the rating lookups flips from (matrix.loc[neighbor, target_item]) to (matrix.loc[target_user, neighbor_item]).

# Part E - Evaluation

In [ ]:
# ===== AI-ASSISTED START (Claude): train/test split =====
def train_test_split_ratings(ratings_df, test_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    test_idx = []
    for user, grp in ratings_df.groupby("userId"):
        n = len(grp)
        n_test = int(round(n * test_fraction))
        n_test = min(n_test, n - 1)  # always keep >= 1 rating in train
        if n_test <= 0:
            continue
        test_idx.extend(rng.choice(grp.index.to_numpy(), size=n_test, replace=False))
    test_df  = ratings_df.loc[test_idx]
    train_df = ratings_df.drop(index=test_idx)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, test_df = train_test_split_ratings(ratings_df, 0.2)
print(len(train_df), len(test_df), len(train_df) + len(test_df) == len(ratings_df))
# own check: every user still has train ratings, and ~20% of each user's ratings are in test
print((test_df.groupby("userId").size() / ratings_df.groupby("userId").size()).describe())
print(set(train_df.userId) == set(ratings_df.userId))

train_matrix = train_df.pivot_table(index="userId", columns="movieId", values="rating")
# ===== AI-ASSISTED END =====

In [ ]:
def evaluate(matrix, test_df, k, similarity_fn, method, max_test=None, seed=42):
    #mtnsash en method: "user" or "item". Returns dict(rmse, mae, frac_none, n_pred, n_total).
    predict = {"user": predict_rating_user_based, "item": predict_rating_item_based}[method]
    rows = test_df
    if max_test is not None and len(test_df) > max_test: # optional speed-up: fixed random subset
        rows = test_df.sample(max_test, random_state=seed)
    errors, n_none = [], 0
    for u, i, r in zip(rows["userId"], rows["movieId"], rows["rating"]):
        if u not in matrix.index or i not in matrix.columns:  # unseen user/item in train -> impossible
            n_none += 1
            continue
        p = predict(matrix, u, i, k, similarity_fn)
        if p is None:
            n_none += 1
        else:
            errors.append(p - r)
    errors = np.array(errors)
    n_total = len(rows)
    if len(errors) == 0:
        return {"rmse": np.nan, "mae": np.nan, "frac_none": 1.0, "n_pred": 0, "n_total": n_total}
    return {"rmse": float(np.sqrt(np.mean(errors ** 2))),
            "mae": float(np.mean(np.abs(errors))),
            "frac_none": n_none / n_total,
            "n_pred": len(errors), "n_total": n_total}

def run_grid(matrix, test_df, ks=(5, 10, 20, 50), max_test=None):
    sims = {"euclidean": euclidean_similarity, "cosine": cosine_similarity, "pearson": pearson_similarity}
    out = []
    for method in ["user", "item"]:
        for name, fn in sims.items():
            for k in ks:
                res = evaluate(matrix, test_df, k, fn, method, max_test=max_test)
                out.append({"method": method + "-based", "similarity": name, "k": k, **res})
                print(out[-1])
    return pd.DataFrame(out)

print(evaluate(train_matrix, test_df, 10, pearson_similarity, "user", max_test=200))
print(evaluate(train_matrix, test_df, 10, pearson_similarity, "item", max_test=200))

In [ ]:
import time
M = train_matrix.to_numpy(); mask = ~np.isnan(M)
MT, maskT = M.T.copy(), mask.T.copy()
uidx = {u: i for i, u in enumerate(train_matrix.index)}
iidx = {m: i for i, m in enumerate(train_matrix.columns)}

def _sims(X, Xm, a, metric):
    co = Xm & Xm[a][None, :]                      # co-rated entries only
    n = co.sum(1)
    Xc = np.where(co, X, 0.0); xa = np.where(co, X[a][None, :], 0.0)
    if metric == "euclidean":
        s = 1 / (1 + np.sqrt(((Xc - xa) ** 2).sum(1)))
    else:
        if metric == "pearson":                   # mean-center over co-rated items
            nn = np.maximum(n, 1)
            Xc = np.where(co, X - (Xc.sum(1) / nn)[:, None], 0.0)
            xa = np.where(co, X[a][None, :] - (xa.sum(1) / nn)[:, None], 0.0)
        den = np.linalg.norm(Xc, axis=1) * np.linalg.norm(xa, axis=1)
        s = np.where(den == 0, 0.0, (Xc * xa).sum(1) / np.where(den == 0, 1, den))
    ok = n >= 2; ok[a] = False                    # need >= 2 co-rated, skip itself
    return s, ok

_cache = {}
def _neighbors(kind, idx, metric):
    key = (kind, idx, metric)
    if key not in _cache:
        X, Xm = (M, mask) if kind == "user" else (MT, maskT)
        s, ok = _sims(X, Xm, idx, metric)
        cand = np.where(ok)[0]
        _cache[key] = (cand[np.argsort(-s[cand], kind="stable")], s)
    return _cache[key]

def fast_predict(kind, u, i, k, metric):
    if u not in uidx or i not in iidx:
        return None
    ui, ii = uidx[u], iidx[i]
    if kind == "user":
        order, s = _neighbors("user", ui, metric); nb = order[:k]
        m = mask[nb, ii]; w = s[nb][m]; r = M[nb, ii][m]
    else:
        order, s = _neighbors("item", ii, metric); nb = order[:k]
        m = mask[ui, nb]; w = s[nb][m]; r = M[ui, nb][m]
    den = np.abs(w).sum()
    return None if den == 0 else float((w * r).sum() / den)

# validation against the pandas functions above (user-based all metrics, item-based euclidean)
chk = test_df.sample(15, random_state=1); bad = 0
sims = {"euclidean": euclidean_similarity, "cosine": cosine_similarity, "pearson": pearson_similarity}
for u, i in zip(chk.userId, chk.movieId):
    for name, fn in sims.items():
        a = predict_rating_user_based(train_matrix, u, i, 20, fn); b = fast_predict("user", u, i, 20, name)
        bad += (a is None) != (b is None) or (a is not None and abs(a - b) > 1e-6)
for u, i in zip(chk.userId[:3], chk.movieId[:3]):
    a = predict_rating_item_based(train_matrix, u, i, 20, euclidean_similarity); b = fast_predict("item", u, i, 20, "euclidean")
    bad += (a is None) != (b is None) or (a is not None and abs(a - b) > 1e-6)
print("mismatches vs reference functions:", bad)   # expect 0

In [ ]:
rows = test_df.sample(2000, random_state=42)
res = []
for kind in ("user", "item"):
    for metric in ("euclidean", "cosine", "pearson"):
        for k in (5, 10, 20, 50):
            errs, none = [], 0
            for u, i, r in zip(rows.userId, rows.movieId, rows.rating):
                p = fast_predict(kind, u, i, k, metric)
                if p is None: none += 1
                else: errs.append(p - r)
            e = np.array(errs)
            res.append({"method": kind + "-based", "similarity": metric, "k": k,
                        "rmse": np.sqrt((e**2).mean()), "mae": np.abs(e).mean(),
                        "frac_none": none / len(rows)})
        _cache.clear()
results = pd.DataFrame(res)
for v in ("rmse", "mae", "frac_none"):
    print(v); display(results.pivot_table(index=["method", "similarity"], columns="k", values=v).round(3))

# Part F

## 16. 
Best: item-based, Euclidean, k=50 (RMSE 1.038, MAE 0.768, 37.6% of ratings had no prediction).
Worst: user-based, Pearson, k=5 (RMSE 1.294).

This does NOT match the Section 3.2 intuition. Section 3.2 predicts Pearson > cosine > Euclidean. In my results Pearson was the worst metric in both CF directions (item: 1.18-1.25, user: 1.17-1.29), cosine was in the middle, and Euclidean was best or equal best (item k=50: 1.038 vs cosine 1.116 vs Pearson 1.180).

Possible reasons:
- With co-rated-only data, two items/users sharing just 2 ratings get Pearson = +1 or -1 exactly, so the top-k neighbours are filled with tiny-overlap ties, which makes Pearson noisy.
- The RMSEs are not on identical rows: frac_none differs per cell (36%-91%), because the k neighbours are chosen among all users/items and then only those that rated the target are used. At small k most predictions are impossible.
- Euclidean (1/(1+d)) is high for pairs with few co-rated items, but the prediction averages raw ratings, so such neighbours mostly give "typical" ratings, which keeps RMSE low.

Trends: RMSE and frac_none both decrease as k grows. Item-based had less missing coverage (0.36-0.71 vs 0.51-0.91) and lower RMSE in most cells.


## 17. Example

![alt text](download.png)

According to the calculations of the example.. Eclidean Similarity shows that the two users are close to be similar (0.536). While Pearson's Corelation shows that the two users are completly opposite (-1).
They disagree because Eclidean similarity depends on gaps only, while pearson compares the two users with their own averages.

## 18. Comparison between user-based CF and item-based CF

The user-based collaborative filtering is nearly impossible to be up to date in large systems, because as in the example, each target user is compared with 200 million other users, so if we needed to compare all users its (200M x 200M) = 2x(10^16) comparisons, this is not efficient at all and very expensive.

On the other hand, the item-based collaborative filtering is very convinient in comparison with the user-based, as in the example, with only 20,000 products, if we wanted to compare each target item with all the other items we will get (20K x 20K) = 2x(10^8) comparisons only, this is much cheaper and efficient to stay up to date. 

# Bonus

## Mean-centring

In [1]:
def predict_rating_user_based_centered(matrix, target_user, target_item, k, similarity_fn):
    if target_item not in matrix.columns or target_user not in matrix.index:
        return None
    neighbors = find_k_nearest_users(matrix, target_user, k, similarity_fn)
    target_mean = matrix.loc[target_user].mean()
    numerator = 0
    denominator = 0
    for user, score in neighbors:
        rating = matrix.loc[user, target_item]
        if np.isnan(rating):
            continue
        neighbor_mean = matrix.loc[user].mean()
        numerator += score * (rating - neighbor_mean)
        denominator += abs(score)
    if denominator == 0:
        return None
    return target_mean + (numerator / denominator)


### Comparing RMSE with plain Pearson-weighted prediction
For 200 random ratings, I hide the true rating in a copy of the matrix, predict it with both methods (k=20, Pearson), then compare RMSE. Both predictors use the same neighbours, so they return None in exactly the same cases.

In [2]:
sample = ratings_df.sample(200, random_state=42)

errors_plain, errors_centered = [], []
none_plain = none_centered = 0

for _, row in sample.iterrows():
    u, i, true = int(row["userId"]), int(row["movieId"]), row["rating"]
    matrix.loc[u, i] = np.nan                        # hide the true rating
    p1 = predict_rating_user_based(matrix, u, i, 20, pearson_similarity)
    p2 = predict_rating_user_based_centered(matrix, u, i, 20, pearson_similarity)
    matrix.loc[u, i] = true                          # restore it

    if p1 is None:
        none_plain += 1
    else:
        errors_plain.append(p1 - true)

    if p2 is None:
        none_centered += 1
    else:
        errors_centered.append(p2 - true)

rmse_plain = np.sqrt(np.mean(np.square(errors_plain)))
rmse_centered = np.sqrt(np.mean(np.square(errors_centered)))

print("Plain    RMSE:", rmse_plain, "  None:", none_plain)
print("Centered RMSE:", rmse_centered, "  None:", none_centered)

NameError: name 'ratings_df' is not defined

Both predictors use the same neighbours and the same Pearson weights, because both call find_k_nearest_users, so they return None in exactly the same cases (<none count> of 200). Pearson already mean-centers the ratings inside the similarity, so it finds users with the same taste pattern. The plain predictor still averages the neighbours' raw ratings, so a generous neighbour pulls the prediction up. The centered predictor fixes that by subtracting each neighbour's mean rating and adding back the target user's mean. The two give the same prediction when the neighbours' means equal the target user's mean, which is why they are closely related. They are not identical, because Pearson centers over the co-rated items only, while the predictor centers over each user's whole rating history.

Results: plain RMSE = <value>, centered RMSE = <value>, computed on <n> predictions. <Write one sentence: which is lower, and whether the gap is large or small given only about <n> usable predictions.>

## Bonus - Top-10 recommendations for one user
Predicted ratings for all items the user has not rated, using user-based CF with Pearson (k=50), then a manual genre check.

In [ ]:
def top_n_recommendations(matrix, user, movies_df, k=50, n=10, similarity_fn=pearson_similarity, min_neighbors=3):
    # 1) k most similar users (co-rated >= 2)
    sims = []
    for other in matrix.index:
        if other == user:
            continue
        v1, v2 = get_co_rated(matrix, user, other, "index")
        if len(v1) < 2:
            continue
        sims.append((other, similarity_fn(v1, v2)))
    sims.sort(key=lambda x: x[1], reverse=True)
    sims = sims[:k]
    ids = [s[0] for s in sims]
    w = np.array([s[1] for s in sims])

    # 2) similarity-weighted average per item, using only neighbours who rated that item (Section 3.5)
    R = matrix.loc[ids]
    mask = R.notna().to_numpy()
    vals = R.fillna(0).to_numpy() # the zeros are multiplied by mask, they never count as ratings
    num = (w[:, None] * vals * mask).sum(axis=0)
    den = (np.abs(w)[:, None] * mask).sum(axis=0)
    cnt = mask.sum(axis=0)
    pred = np.where((cnt >= min_neighbors) & (den > 0), num / np.where(den == 0, 1, den), np.nan)
    pred = pd.Series(pred, index=matrix.columns)

    # 3) only items the user has not rated
    pred = pred[matrix.loc[user].isna()].dropna().sort_values(ascending=False).head(n)
    out = pred.rename("predicted_rating").reset_index().merge(movies_df, on="movieId")
    return out

user = 1   
recs = top_n_recommendations(matrix, user, movies_df)
display(recs[["title", "genres", "predicted_rating"]])

# Manual sanity check: what does this user actually like?
hist = ratings_df[ratings_df.userId == user].merge(movies_df, on="movieId").sort_values("rating", ascending=False)
display(hist.head(10)[["title", "genres", "rating"]])
print(hist[hist.rating >= 4].genres.str.split("|").explode().value_counts().head(5))
print(recs.genres.str.split("|").explode().value_counts().head(5))

Genre check for user 1: the user's 4+ ratings are mostly Action (76), Adventure (74), Comedy (70), Drama (64) and Thriller (43). The recommendations are mostly Drama (8), Crime (4), Comedy (3), Romance (3) and Thriller (2). Drama, Comedy and Thriller match, but Action and Adventure, which this user rates highly most often, are barely represented, so the match is only partial. The list is also dominated by well-known classics (The Godfather, The Shawshank Redemption), probably because popular movies have enough neighbour ratings to pass min_neighbors.